## Check the scripts for cross power spectrum

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import lsstypes as types

os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'

sys.path.insert(0, '../../')
from cat_tools import get_measurement_fn, get_measurement_ready_fn
from fit_tools import read_data_from_fn
from helper import GET_REDSHIFT_SET, SNAP_TO_ZP

%load_ext autoreload
%autoreload 2

In [ ]:
corr = 'mesh2'
version ='AbacusHF-v2'
tracer = 'LRGxELG'
zsnaps, zranges = GET_REDSHIFT_SET(version, 'altmtl')
zrange = (0.8, 1.1)  # Match the saved cross spectrum and our example run.
region = 'GCcomb'
use_dv = False  # --zerrs None

data_args = {'version':version, 'domain':'altmtl', 'tracer':tracer, 'zrange':zrange, 'mock_id': 0, 'use_dv':use_dv, 'region':region}

get_measurement_fn(**data_args, mkdir=False)

In [ ]:
mocks = 'abacus-hf-dr2-v2-altmtl'
cross_fn = '/global/cfs/cdirs/desi/science/cai/desi-clustering/dr2/summary_statistics/full_shape/multitracer/abacus-hf-dr2-v2-altmtl/mesh2_spectrum_poles_LRGxELG_LOPnotqso_z0.8-1.1_GCcomb_weight-default-FKP_0.h5'
cov_fn = '/global/cfs/cdirs/desi/science/cai/desi-clustering/dr2/summary_statistics/full_shape/multitracer/holi-v3-altmtl/covmat_mesh2_spectrum_LL_EE_LE_983mocks_GCcomb.h5'

data = types.read(cross_fn)
data = data.select(k=slice(0, None, 5)).select(k=(0.02, 0.4))
k = data.get(ells=0).coords('k')
Pk = [data.get(ells=ell).values()['value'] for ell in data.ells]
cov=types.read(cov_fn)
cov = cov.at.observable.get(observables='LE')
cov = cov.at.observable.match(data)
err = np.asarray(cov.std())
errs = np.split(err, 3)

In [ ]:
# Use the retained reference-mesh test; the old production measurements were removed.
from pathlib import Path
measurement_dir = Path('../results/cross_mesh750_nran15').resolve()
measured_fn = get_measurement_fn(**data_args, base_dir=measurement_dir, mkdir=False).format('mesh2_spectrum_poles')
measured = types.read(measured_fn).match(data)
Pk_measured = [measured.get(ells=ell).values()['value'] for ell in data.ells]
measured_fn


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 6), sharex='col', constrained_layout=True,
                         gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.08})
for i, (ell, pk, pk_measured, err) in enumerate(zip(data.ells, Pk, Pk_measured, errs)):
    ax, ax_res = axes[:, i]
    k = data.get(ells=ell).coords('k')
    k_measured = measured.get(ells=ell).coords('k')
    ax.errorbar(k, k * pk, yerr=k * err, fmt='.', capsize=2,
                label='Saved (Holi mock errors)')
    ax.plot(k_measured, k_measured * pk_measured, label='Reference mesh, randoms 10/15')
    ax.set_title(fr'$\ell = {ell}$')
    ax.set_ylabel(r'$k P_\ell(k)\ [(\mathrm{Mpc}/h)^2]$')
    ax.grid(alpha=0.2)

    # Normalize by the single-realization covariance error, not an error on the difference.
    residual = (np.asarray(pk_measured) - np.asarray(pk)) / err
    ax_res.axhline(0, color='0.3', ls='--', lw=1)
    ax_res.plot(k, residual, '.-', color='C1', ms=3)
    ax_res.set_xlabel(r'$k\ [h\,\mathrm{Mpc}^{-1}]$')
    ax_res.set_ylabel(r'$\Delta P_\ell / \sigma_\ell$')
    ax_res.grid(alpha=0.2)
axes[0, 0].legend()
fig.suptitle(f'{tracer}, {zrange[0]} < z < {zrange[1]}, {region}, mock {data_args["mock_id"]}')
plt.show()